# 07 — Detecção de mudança com Fourier (proposta do professor)

**Pergunta:** comparar os espectros de Fourier de dois anos acha onde a paisagem mudou melhor do que
a diferença simples de um índice espectral?

**Setup**
- Dados: os 12 recortes de `amazonia_exploracao` (256×256 px, 10 m/px, 6 bandas Sentinel-2).
- Par de anos: **t1 = 2019** e **t2 = 2025** (as duas pontas do que baixamos).
- **Gabarito:** pixel cuja classe do MapBiomas em 2019 é diferente da de 2025.
- Nenhum método aqui é treinado: cada um é uma fórmula que recebe (t1, t2) e devolve um
  **mapa de pontuação** (alto = "acho que mudou aqui").

**Roteiro**
1. Dados, índices (NDVI, BSI) e gabarito
2. Intuição de Fourier nos nossos dados
3. Os métodos: baseline, Teste 1, controle, Teste 2
4. Mapas lado a lado
5. Validação (AUC)
6. Conclusões

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from utils.download_ee import carregar_amostra

AMOSTRA = 'amazonia_exploracao'
T1, T2 = 2019, 2025

## 1. Dados, índices e gabarito

Índices espectrais (um número por pixel, calculado a partir das bandas):
- **NDVI** = (B8 − B4) / (B8 + B4): alto onde há **vegetação**.
- **BSI** = ((B11 + B4) − (B8 + B2)) / ((B11 + B4) + (B8 + B2)): alto onde há **solo exposto**
  (mineração, estrada, terra arada).

In [ ]:
X, mapbiomas, anos, info = carregar_amostra(AMOSTRA)
BANDA = {b: i for i, b in enumerate(info.attrs['bandas'])}
i1, i2 = anos.index(T1), anos.index(T2)

refl = X.astype(np.float32) / 10000  # reflectância de superfície em [0, 1]


def ndvi(r):
    b8, b4 = r[..., BANDA['B8']], r[..., BANDA['B4']]
    return (b8 - b4) / (b8 + b4 + 1e-6)


def bsi(r):
    b2, b4, b8, b11 = (r[..., BANDA[b]] for b in ('B2', 'B4', 'B8', 'B11'))
    return ((b11 + b4) - (b8 + b2)) / ((b11 + b4) + (b8 + b2) + 1e-6)


# INDICES[nome] tem shape (recortes, 2, H, W): posição 0 = t1, posição 1 = t2
# (float64: a FFT em float32 perde precisão nas frequências de magnitude pequena)
INDICES = {'NDVI': ndvi(refl[:, [i1, i2]]).astype(np.float64),
           'BSI': bsi(refl[:, [i1, i2]]).astype(np.float64)}

# Gabarito: a classe do MapBiomas mudou entre t1 e t2 (classe 0 = sem dado, fica de fora)
mb1, mb2 = mapbiomas[:, i1], mapbiomas[:, i2]
valido = (mb1 > 0) & (mb2 > 0)
gabarito = (mb1 != mb2) & valido

info['% mudou'] = (100 * gabarito.sum((1, 2)) / valido.sum((1, 2))).round(2)
info[['arquivo', 'criterio', 'ecorregiao', '% mudou']]

Os recortes `uniforme` quase não mudam (alguns têm 0%). Sem pixels que mudaram não existe AUC, então
na validação só entram recortes com **pelo menos 1% de mudança**.

Atenção: o critério `mudou` do sorteio olhava **todos** os anos, então um recorte `mudou` pode ter
pouca diferença entre 2019 e 2025 especificamente.

## 2. Intuição de Fourier nos nossos dados

A FFT (`np.fft.fft2`) transforma a imagem numa soma de ondas. Cada onda tem:
- **magnitude**: quanto daquela onda existe (que texturas a imagem tem);
- **fase**: onde a onda está posicionada (onde as coisas ficam).

Com `fftshift`, o **centro** do espectro são as frequências baixas (áreas grandes e uniformes,
brilho médio) e as **bordas** são as frequências altas (contornos, estradas, detalhes finos).

In [ ]:
def mostrar(ax, img, titulo, cmap='gray', clip=99):
    lo, hi = np.percentile(img, [100 - clip, clip])
    ax.imshow(img, cmap=cmap, vmin=lo, vmax=hi)
    ax.set_title(titulo, fontsize=9)
    ax.axis('off')


def rgb(n, i_ano, ganho=3.5):
    return np.clip(refl[n, i_ano][..., [BANDA['B4'], BANDA['B3'], BANDA['B2']]] * ganho, 0, 1)


N_EX = 7  # recorte de mineração usado como exemplo
img = INDICES['NDVI'][N_EX, 1]
F = np.fft.fftshift(np.fft.fft2(img))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
axes[0].imshow(rgb(N_EX, i2)); axes[0].set_title(f'RGB {T2}', fontsize=9); axes[0].axis('off')
mostrar(axes[1], img, f'NDVI {T2}', cmap='RdYlGn')
mostrar(axes[2], np.log(np.abs(F) + 1e-8), 'log-magnitude (centro = baixa freq.)', cmap='magma')
mostrar(axes[3], np.angle(F), 'fase', cmap='twilight', clip=100)
plt.tight_layout(); plt.show()

A fase parece ruído, mas é ela que guarda **onde** as coisas estão. Experimento clássico: juntar a
**magnitude de um recorte com a fase de outro** e voltar para imagem.

In [ ]:
def trocar(a, b):
    """Imagem com a magnitude de `a` e a fase de `b`."""
    Fa, Fb = np.fft.fft2(a), np.fft.fft2(b)
    return np.real(np.fft.ifft2(np.abs(Fa) * np.exp(1j * np.angle(Fb))))


A, B = INDICES['NDVI'][7, 1], INDICES['NDVI'][11, 1]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
mostrar(axes[0], A, 'A')
mostrar(axes[1], B, 'B')
mostrar(axes[2], trocar(A, B), 'magnitude de A + fase de B')
mostrar(axes[3], trocar(B, A), 'magnitude de B + fase de A')
plt.tight_layout(); plt.show()

Cada imagem trocada se parece com a dona da **fase**: a posição das estruturas mora na fase.

O outro argumento do professor: **deslocar a imagem não muda a magnitude**, só a fase. Por isso
comparar magnitudes tolera desalinhamento entre os anos.

In [ ]:
deslocada = np.roll(A, shift=(2, 2), axis=(0, 1))  # 2 pixels para baixo e para a direita
mag = lambda a: np.abs(np.fft.fft2(a))
fase = lambda a: np.angle(np.fft.fft2(a))
print('magnitude igual depois de deslocar?', np.allclose(mag(A), mag(deslocada)))
print('fase igual depois de deslocar?     ', np.allclose(fase(A), fase(deslocada)))

## 3. Os métodos

Todos recebem duas imagens de índice (`a1` = t1, `a2` = t2) e devolvem um mapa 256×256.

| Método | Ideia |
|---|---|
| **Baseline** | `|a2 − a1|` pixel a pixel |
| **Teste 1** | resíduo do log-espectro, reconstruído com a fase de t2 (receita do professor) |
| **Teste 1 controle** | Teste 1 com t2 contra ele mesmo: **nada mudou**, então deveria dar mapa vazio |
| **Teste 2** | FFT em blocos 64×64 com janela de Hann; energia da diferença das magnitudes em alta frequência |

### Teste 1: resíduo espectral

Passos do professor: `L = log|F|` dos dois anos, `residuo = L2 − L1`, volta para imagem com a fase
de t2. Algebricamente isso é `|ifft(F2 / |F1|)|²`: a imagem de t2 **"branqueada" pelo espectro de
t1**. Ondas que ficaram mais fortes em t2 ganham destaque.

Duas variantes opcionais:
- `raio_centro`: zera o resíduo nas frequências mais baixas (ignora brilho geral e iluminação).
- `sigma`: suaviza o resíduo (a divisão por `|F1|` amplifica ruído onde `|F1|` é quase zero).

In [ ]:
def raio_frequencia(shape):
    """Distância de cada posição do espectro (sem shift) até a frequência zero, em ciclos/pixel."""
    fy, fx = np.meshgrid(np.fft.fftfreq(shape[0]), np.fft.fftfreq(shape[1]), indexing='ij')
    return np.hypot(fy, fx)


def baseline(a1, a2):
    return np.abs(a2 - a1)


def teste1(a1, a2, raio_centro=0.0, sigma=0.0):
    F1, F2 = np.fft.fft2(a1), np.fft.fft2(a2)
    residuo = np.log(np.abs(F2) + 1e-8) - np.log(np.abs(F1) + 1e-8)
    if sigma:
        residuo = gaussian_filter(residuo, sigma, mode='wrap')  # o espectro é periódico
    if raio_centro:
        residuo[raio_frequencia(a1.shape) <= raio_centro] = 0
    return np.abs(np.fft.ifft2(np.exp(residuo + 1j * np.angle(F2)))) ** 2


def controle(a1, a2):
    return teste1(a2, a2)

### Teste 2: FFT em blocos

1. Percorre a imagem em blocos de 64×64 com passo 32 (sobreposição de 50%, para uma mudança na
   divisa entre dois blocos não sumir).
2. Multiplica cada bloco pela **janela de Hann 2D** (bordas do bloco indo suavemente a zero, o
   "tilt-shift matemático": evita a borda falsa que a FFT vê quando o bloco não é periódico).
3. Energia da mudança = soma de `(|F2| − |F1|)²` só nas **altas frequências** (raio > 0,125).
4. Cada pixel recebe a média da energia dos blocos que o cobrem.

In [ ]:
BLOCO, PASSO, CORTE_ALTA = 64, 32, 0.125
HANN = np.outer(np.hanning(BLOCO), np.hanning(BLOCO))
ALTA = raio_frequencia((BLOCO, BLOCO)) > CORTE_ALTA


def teste2(a1, a2):
    soma, cont = np.zeros_like(a1), np.zeros_like(a1)
    for y in range(0, a1.shape[0] - BLOCO + 1, PASSO):
        for x in range(0, a1.shape[1] - BLOCO + 1, PASSO):
            janela = np.s_[y:y + BLOCO, x:x + BLOCO]
            m1 = np.abs(np.fft.fft2(a1[janela] * HANN))
            m2 = np.abs(np.fft.fft2(a2[janela] * HANN))
            soma[janela] += ((m2 - m1)[ALTA] ** 2).sum()
            cont[janela] += 1
    return soma / cont


METODOS = {
    'Baseline |Δ|': baseline,
    'Teste 1': teste1,
    'Teste 1 (centro zerado)': lambda a1, a2: teste1(a1, a2, raio_centro=0.02),
    'Teste 1 (suavizado)': lambda a1, a2: teste1(a1, a2, sigma=3),
    'Teste 1 controle (t2 vs t2)': controle,
    'Teste 2 (blocos)': teste2,
}

# MAPAS[(índice, método)] tem shape (recortes, H, W)
MAPAS = {(ind, nome): np.stack([f(a[0], a[1]) for a in INDICES[ind]])
         for ind in INDICES for nome, f in METODOS.items()}

### Máscara central (o "tilt-shift" aplicado no final)

O professor sugere multiplicar o mapa final por uma Gaussiana centrada, para focar no meio.

**Cuidado:** nos recortes `mineracao` e `mudou`, o ponto sorteado fica **no centro por construção**.
A máscara central pode melhorar o resultado só por imitar o jeito como sorteamos, um viés que não
existe num uso real. Por isso a validação mostra os métodos **com e sem** a máscara.

In [ ]:
yy, xx = np.mgrid[:256, :256] - 127.5
MASCARA_CENTRAL = np.exp(-(yy ** 2 + xx ** 2) / (2 * 64 ** 2))  # sigma de 64 px

plt.figure(figsize=(3, 3))
plt.imshow(MASCARA_CENTRAL, cmap='gray'); plt.title('máscara central (σ = 64 px)', fontsize=9)
plt.axis('off'); plt.show()

## 4. Mapas lado a lado (NDVI)

Uma linha por recorte. Nos mapas, **claro = pontuação alta** ("mudou"). A escala de cada painel vai
do percentil 1 ao 99 do próprio mapa (só o ranking importa, não o valor absoluto).

In [ ]:
COLUNAS = ['Baseline |Δ|', 'Teste 1', 'Teste 1 controle (t2 vs t2)', 'Teste 2 (blocos)']


def figura_mapas(indice='NDVI', recortes=range(len(X))):
    recortes = list(recortes)
    fig, axes = plt.subplots(len(recortes), 3 + len(COLUNAS), figsize=(17, 2.5 * len(recortes)))
    for linha, n in zip(axes, recortes):
        linha[0].imshow(rgb(n, i1)); linha[1].imshow(rgb(n, i2))
        linha[2].imshow(gabarito[n], cmap='gray')
        for ax in linha[:3]:
            ax.axis('off')
        for ax, nome in zip(linha[3:], COLUNAS):
            mostrar(ax, MAPAS[(indice, nome)][n], '', cmap='inferno')
        linha[0].set_title(f'{info.arquivo[n]}\nRGB {T1}', fontsize=8)
    for ax, t in zip(axes[0, 1:], [f'RGB {T2}', 'gabarito (MapBiomas mudou)'] + COLUNAS):
        ax.set_title(t, fontsize=8)
    plt.tight_layout(); plt.show()


figura_mapas('NDVI')

## 5. Validação: AUC contra o gabarito

Para cada recorte e cada mapa: **AUC** = probabilidade de um pixel que mudou ter pontuação maior que
um pixel que não mudou. 0,5 = chute, 1,0 = perfeito. Não precisa escolher limiar.

A AUC é calculada **por recorte** (a escala do Teste 1 muda de imagem para imagem, então misturar
pixels de recortes diferentes seria injusto) e depois tiramos a média. Entram só os recortes com
pelo menos 1% de mudança.

In [ ]:
AVALIADOS = [n for n in range(len(X)) if info['% mudou'][n] >= 1]


def auc(mapa, n):
    return roc_auc_score(gabarito[n][valido[n]], mapa[valido[n]])


linhas = []
for (ind, nome), mapas in MAPAS.items():
    for n in AVALIADOS:
        linhas.append({'índice': ind, 'método': nome, 'recorte': info.arquivo[n],
                       'sem máscara': auc(mapas[n], n),
                       'com máscara central': auc(mapas[n] * MASCARA_CENTRAL, n)})
aucs = pd.DataFrame(linhas)

print('Recortes avaliados:', [info.arquivo[n] for n in AVALIADOS])
(aucs.groupby(['índice', 'método'], sort=False)[['sem máscara', 'com máscara central']]
     .mean().unstack('índice').round(3))

AUC média por método. Abaixo, recorte a recorte (NDVI, sem máscara), para ver se a média esconde
algum caso.

In [ ]:
(aucs[aucs['índice'] == 'NDVI']
     .pivot(index='recorte', columns='método', values='sem máscara')[list(METODOS)].round(3))

In [ ]:
media = aucs.groupby(['índice', 'método'], sort=False)['sem máscara'].mean().unstack('índice')
ax = media.plot.barh(figsize=(8, 4), color=['#2e7d32', '#8d6e63'])
ax.axvline(0.5, color='gray', ls='--', lw=1)
ax.text(0.505, -0.6, 'chute', color='gray', fontsize=8)
ax.set_xlim(0.3, 1); ax.set_xlabel('AUC média (sem máscara)'); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 6. Conclusões

AUC média em 8 recortes, NDVI, sem máscara: **baseline 0,76 · Teste 2 0,67 · Teste 1 0,52 (chute)**.

1. **O Teste 1 não detecta mudança.** Ele fica no nível do chute, e o controle (t2 contra ele
   mesmo, onde nada mudou) tem AUC **igual ou maior** (0,57). Nos mapas, o controle acende as
   **bordas de 2025** (rios, estradas, contorno da mina), e o Teste 1 acrescenta listras, que são
   ruído da divisão por `|F1|`. Motivo: a magnitude é calculada para a imagem inteira e não diz
   onde algo mudou; a posição vem só da fase de t2, que descreve as estruturas de t2, mudadas ou
   não. Zerar o centro não muda nada, e suavizar só aproxima o resultado do controle.
2. **O Teste 2 funciona, mas perde do baseline.** Comparar blocos localiza a mudança (0,86 no
   08_mineracao). Só que o mapa é grosseiro (blocos de 64 px) e mede só textura (alta frequência).
   Quando uma floresta inteira vira pastagem (11_mudou, 87% do recorte), a textura quase não muda
   e ele erra (0,32).
3. **A máscara central piora todos os métodos que funcionam.** Entre 2019 e 2025, a mudança não
   está concentrada no centro (o ponto sorteado mudou em só 3 dos 12 recortes nesse par de anos).
   Foi bom ter testado: com o recorte centrado no evento e o mesmo par de anos usado no sorteio, a
   máscara teria inflado o resultado.
4. **O gabarito tem limites.** No 10_mudou, a "mudança" é quase toda pastagem ↔ mosaico de usos
   (classes 15 e 21 do MapBiomas), uma troca de rótulo entre classes parecidas e não uma mudança
   física. Lá até o baseline fica abaixo de 0,5.
5. **NDVI é um pouco melhor que BSI** em todos os métodos.

**Leitura geral:** a intuição do professor (comparar estrutura, tolerar desalinhamento) só rende
quando a comparação é **local** (Teste 2). Nos nossos dados, os compostos anuais já vêm
co-registrados, então essa tolerância não compra muita coisa, e a diferença simples de índice segue
como o método a bater.

**Próximos passos**
- Combinar baseline + Teste 2 (Δ de nível + Δ de textura) e ver se a soma passa de 0,76.
- Repetir com pares consecutivos (2024→2025), mais próximo do monitoramento real.
- Mais recortes: com 8 avaliados, as diferenças entre métodos ainda são frágeis.
- Colocar os embeddings do TerraMind na mesma tabela, como competidor aprendido.